In [ ]:
import numpy as np
import torch
from torchvision import datasets, transforms
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from optimizer import TrellisOptimizer
from lineart import LinearT
from tqdm import tqdm
import gc
import os

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_dataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset  = datasets.MNIST(root='./data', train=False, transform=transform, download=True)

train_loader = DataLoader(dataset=train_dataset, batch_size=16, shuffle=True)
test_loader  = DataLoader(dataset=test_dataset, batch_size=1000, shuffle=False)

In [4]:
class mnist(nn.Module):
    def __init__(self, block_size):
        super().__init__()
        q, _ = torch.linalg.qr(torch.randn(784, 784, device=device))
        self.register_buffer('q', q)
        self.layer1 = LinearT(784, 1024, block_size=block_size)
        self.relu1 = nn.ReLU()
        self.drop1 = nn.Dropout(0.2)
        self.layer2 = LinearT(1024, 1024, block_size=block_size)
        self.relu2 = nn.ReLU()
        self.classifier = nn.Linear(1024, 10)
        
    def forward(self, x):
        x = x.flatten(start_dim=1)
        x = x @ self.q
        x = self.relu1(self.layer1(x))
        residual = x
        x = self.drop1(x)
        x = self.relu2(self.layer2(x))
        x = x + residual
        return self.classifier(x)

model = mnist(block_size=4).to(device)
criterion = nn.CrossEntropyLoss()

trellis_params = []
adam_params = []

for p in model.parameters():
    if p.requires_grad:
        if getattr(p, 'is_trellis', False):
            trellis_params.append(p)
        else:
            adam_params.append(p)

trellis_optimizer = TrellisOptimizer(trellis_params, lr=0.01, momentum=0.9, num_flips=1)
adam_optimizer = optim.Adam(adam_params, lr=0.01)
gc.collect()

0

In [5]:
epochs = 50
train_losses = []
train_accuracies = []
test_losses = []
test_accuracies = []
checkpoint_path = "mnist_checkpoint.pt"
start_epoch = 0

In [6]:
if os.path.exists(checkpoint_path):
    print(f"Found checkpoint '{checkpoint_path}', resuming training")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    trellis_optimizer.load_state_dict(checkpoint['trellis_optimizer_state_dict'])
    adam_optimizer.load_state_dict(checkpoint['adam_optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    train_losses = checkpoint['train_losses']
    train_accuracies = checkpoint['train_accuracies']
    test_losses = checkpoint['test_losses']
    test_accuracies = checkpoint['test_accuracies']
    print(f"Resuming from loop index {start_epoch}, epoch {start_epoch + 1}")
else:
    print("No checkpoint found, starting training from scratch.")

No checkpoint found, starting training from scratch.


In [7]:
for epoch in range(start_epoch, epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(train_loader)
    for inputs, labels in pbar:
        inputs = inputs.to(device)
        labels = labels.to(device)

        trellis_optimizer.zero_grad()
        adam_optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        trellis_optimizer.step()
        adam_optimizer.step()
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        pbar.set_description(f'loss: {loss.item()}')
    print(f'Epoch [{epoch + 1}/{epochs}]')
    print("Training loss", (running_loss / len(train_loader)))  
    train_losses.append(running_loss / len(train_loader))
    print("Training accuracy", (correct *100/ total),'%')
    train_accuracies.append(correct * 100 / total)

    model.eval()
    running_loss_eval = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss_eval += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print("Testing loss", (running_loss_eval / len(test_loader)))
    test_losses.append(running_loss_eval / len(test_loader))
    print("Testing accuracy", (correct*100 / total),'%')
    test_accuracies.append(correct * 100 / total)

    checkpoint_data = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'trellis_optimizer_state_dict': trellis_optimizer.state_dict(),
        'adam_optimizer_state_dict': adam_optimizer.state_dict(),
        'train_losses': train_losses,
        'train_accuracies': train_accuracies,
        'test_losses': test_losses,
        'test_accuracies': test_accuracies
    }
    tmp_checkpoint_path = checkpoint_path + ".tmp"
    torch.save(checkpoint_data, tmp_checkpoint_path)
    os.replace(tmp_checkpoint_path, checkpoint_path)
    print(f"Checkpoint saved securely at end of epoch {epoch + 1}")
    
    gc.collect()

Epoch [1/50]
Training loss 0.554954682286332
Training accuracy 88.43666666666667 %
Testing loss 0.3157316043972969
Testing accuracy 92.12 %
Checkpoint saved securely at end of epoch 1
Epoch [2/50]
Training loss 0.2915473492903577
Training accuracy 92.74333333333334 %
Testing loss 0.279489329457283
Testing accuracy 92.9 %
Checkpoint saved securely at end of epoch 2
Epoch [3/50]
Training loss 0.2708742338038088
Training accuracy 93.42666666666666 %
Testing loss 0.28465785160660745
Testing accuracy 93.5 %
Checkpoint saved securely at end of epoch 3
Epoch [4/50]
Training loss 0.2486395730718335
Training accuracy 93.815 %
Testing loss 0.3285737976431847
Testing accuracy 92.78 %
Checkpoint saved securely at end of epoch 4
Epoch [5/50]
Training loss 0.23052251087575643
Training accuracy 94.23 %
Testing loss 0.3281201183795929
Testing accuracy 92.86 %
Checkpoint saved securely at end of epoch 5
Epoch [6/50]
Training loss 0.22740358697590757
Training accuracy 94.39333333333333 %
Testing loss 0.

KeyboardInterrupt: 